# Classification Tensorflow examples

In [ ]:
import pandas as pd
import tensorflow as tf
from tensorflow.keras.datasets import mnist
import sys

sys.path.append("..")

from ai_toolkit import (
    ConfigFactory,
    BaseDataset,
    MobileNetV3SmallModel,
    ClassificationModelTrainer, 
)

## Example data

In [ ]:
class ClfImgDataset(BaseDataset):
    """Dataset for digits (MNIST) image multi-class classification task.
    https://www.tensorflow.org/datasets/catalog/mnist
    """

    def __init__(self) -> None:
        """Initialize the clf image dataset."""

        super().__init__()
        self.batch_size = 1024

    def load_data(self) -> None:
        """Load the mnist dataset for classification task."""
        
        (self.X, self.y), (self.X_test, self.y_test) = mnist.load_data()

        # Only use the first 100 samples for training and 100 for testing
        self.X = self.X[:100]
        self.y = self.y[:100]
        self.X_test = self.X_test[:10]
        self.y_test = self.y_test[:10]
    
    def preprocess(self) -> None:
        """Preprocess the dataset."""

        try:
            self._check_data()
            self._preprocess_mnist()

            # Convert target to pandas series
            self.y = pd.Series(self.y)
            self.y_test = pd.Series(self.y_test)

            self.logger.info(
                "Data preprocessing completed",
                n_samples=self.X.shape[0],
                n_features=self.X.shape[1] * self.X.shape[2],
                n_classes=self.y.nunique(),
            )
        except Exception as e:
            self.logger.error("Data preprocessing failed", error=e)
            raise RuntimeError("Data preprocessing failed") from e
    
    def _preprocess_mnist(self):
        """Preprocess the MNIST dataset"""

        # Convert to float32
        self.X = self.X.astype("float32")
        self.X_test = self.X_test.astype("float32")

        # Normalize to [0,1]
        self.X = self.X / 255.0
        self.X_test = self.X_test / 255.0

        # Convert to tensorflow tensors
        self.X = tf.convert_to_tensor(self.X)
        self.X_test = tf.convert_to_tensor(self.X_test)

        # Add channel dimension (grayscale -> RGB)
        self.X = tf.expand_dims(self.X, axis=-1)
        self.X_test = tf.expand_dims(self.X_test, axis=-1)

        # Convert to RGB (repeat grayscale channel 3 times)
        self.X = tf.image.grayscale_to_rgb(self.X)
        self.X_test = tf.image.grayscale_to_rgb(self.X_test)

        # Resize to 224x224 (MobileNetV3 input size)
        self.X = tf.image.resize(self.X, (224, 224))
        self.X_test = tf.image.resize(self.X_test, (224, 224))

In [ ]:
CImgDataset = ClfImgDataset()
CImgDataset.load_data()
CImgDataset.preprocess()
X_img_clf, y_img_clf, X_test_img_clf = CImgDataset.get_data()

## Configuration

In [ ]:
config_factory = ConfigFactory()
config_factory.training.experiment_name = "ai_toolkit_classification_tf_experiment"
config_factory.training.use_smote = False
configs = config_factory.get_config()

config = configs.training

## Training and evaluation

### Train one example model

In [ ]:
base_model = MobileNetV3SmallModel()

# Create a classification model trainer
trainer = ClassificationModelTrainer(
    base_model=base_model,
    config_factory=config_factory,
)

In [ ]:
# Train and optimize the model
best_model, mean_metrics = trainer.train_and_optimize(
    X=X_img_clf, 
    y=y_img_clf, 
)

# Predict on the test set
# y_pred, y_pred_proba = trainer.predict(X_test_img_clf)